# Fill nan coords using county avg coords

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path

In [17]:
data_dir = Path('../datasets\\')

# Define paths to the storm events CSV files
files = [
    data_dir / 'StormEvents_details-ftp_v1.0_d2016_c20250818.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2017_c20250520.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2018_c20250520.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2019_c20250520.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2020_c20251118.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2021_c20250520.csv',
    data_dir / 'StormEvents_details-ftp_v1.0_d2022_c20250721.csv',
]

# Load datasets
print("Loading datasets...")
df_list = []
for file in files:
    # print(f"  Loading {file.name}...")
    df = pd.read_csv(file)
    # print(f"    Shape: {df.shape}")
    df_list.append(df)

# Concatenate the datasets
df = pd.concat(df_list, ignore_index=True)
print(f"\nCombined dataset shape: {df.shape}")

Loading datasets...

Combined dataset shape: (436146, 51)


In [18]:
df['BEGIN_LAT'].isna().sum(), df['BEGIN_LON'].isna().sum()

(np.int64(167981), np.int64(167981))

In [19]:
# for each row with missing BEGIN_LAT or BEGIN_LON, get them as mean of rows with the same STATE_FIPS
df['BEGIN_LAT'] = df.groupby('STATE_FIPS')['BEGIN_LAT'].transform(
    lambda x: x.fillna(x.mean()).round(4)
)
df['BEGIN_LON'] = df.groupby('STATE_FIPS')['BEGIN_LON'].transform(
    lambda x: x.fillna(x.mean()).round(4)
)

In [20]:
df.drop(columns=[col for col in df.columns if col not in ['INDEX', 'BEGIN_LAT', 'BEGIN_LON']], inplace=True)
df.head()

,BEGIN_LAT,BEGIN_LON
0,34.94,-81.03
1,35.01,-80.93
2,35.64,-82.14
3,35.65,-84.18
4,35.87,-83.77


In [21]:
df.to_csv(data_dir / 'disasters_new_feats' / 'loc.csv', index=False)